In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../datasets/nowplaying_procesed/raw/nowplaying.csv", on_bad_lines='skip', sep="\t")

In [ ]:
import argparse
import time
import csv
import pickle
import operator
import datetime
import os
dataset = '../datasets/nowplaying_procesed/raw/nowplaying.csv'

print("-- Starting @ %ss" % datetime.datetime.now())
with open(dataset, "r") as f:
    reader = csv.DictReader(f, delimiter='\t')
    sess_clicks = {}
    sess_times = {}
    sess_date = {}
    ctr = 0
    curid = -1
    curdate = None
    for data in reader:
        sessid = int(data['SessionId'])
        if curdate and not curid == sessid:
            date = curdate
            sess_date[curid] = date
        curid = sessid

        item = int(data['ItemId'])
        curdate = float(data['Time'])

        if sessid in sess_clicks:
            sess_clicks[sessid] += [item]
            sess_times[sessid] += [curdate]
        else:
            sess_clicks[sessid] = [item]
            sess_times[sessid] = [curdate]
        ctr += 1
    date = float(data['Time'])
    sess_date[curid] = date
print('ctr:', ctr)
print("-- Reading data @ %ss" % datetime.datetime.now())

# Filter out length 1 sessions
for s in list(sess_clicks):
    if len(sess_clicks[s]) == 1:
        del sess_clicks[s]
        del sess_date[s]
        del sess_times[s]

# Count number of times each item appears
iid_counts = {}
for s in sess_clicks:
    seq = sess_clicks[s]
    for iid in seq:
        if iid in iid_counts:
            iid_counts[iid] += 1
        else:
            iid_counts[iid] = 1

sorted_counts = sorted(iid_counts.items(), key=operator.itemgetter(1))

for s in list(sess_clicks):
    curseq = sess_clicks[s]
    curtime = sess_times[s]
    
    filseq = []
    filtime = []

    # 1. Iterar sobre los índices de la secuencia
    for i in range(len(curseq)):
      item_id = curseq[i]
      
      # 2. Aplicar la condición (El item debe aparecer 5 o más veces)
      if iid_counts[item_id] >= 5:
          # Si el item pasa el filtro, lo guardamos
          filseq.append(item_id)
          # Y OBLIGATORIAMENTE guardamos su tiempo asociado
          filtime.append(curtime[i])

    if len(filseq) < 2 or len(filseq) > 30:
        del sess_clicks[s]
        del sess_date[s]
        del sess_times[s]
    else:
        sess_clicks[s] = filseq
        sess_times[s] = filtime

# length = len(sess_clicks)
# for s in list(sess_clicks):
#     curseq = sess_clicks[s]
#     filseq = list(filter(lambda i: iid_counts[i] >= 5, curseq))
#     if len(filseq) < 2 or len(filseq) > 30:
#         del sess_clicks[s]
#         del sess_date[s]
#         del sess_times[s]
#     else:
#         sess_clicks[s] = filseq
#         sess_times[s] = filseq

# Split out test set based on dates
dates = list(sess_date.items())
maxdate = dates[0][1]

for _, date in dates:
    if maxdate < date:
        maxdate = date

# Two months for test
splitdate = maxdate - 60 * 86400

print('Splitting date', splitdate)      # Yoochoose: ('Split date', 1411930799.0)
tra_sess = filter(lambda x: x[1] < splitdate, dates)
tes_sess = filter(lambda x: x[1] > splitdate, dates)

# Sort sessions by date
tra_sess = sorted(tra_sess, key=operator.itemgetter(1))     # [(session_id, timestamp), (), ]
tes_sess = sorted(tes_sess, key=operator.itemgetter(1))     # [(session_id, timestamp), (), ]
print(len(tra_sess))    # 186670    # 7966257
print(len(tes_sess))    # 15979     # 15324
print(tra_sess[:3])
print(tes_sess[:3])
print("-- Splitting train set and test set @ %ss" % datetime.datetime.now())

# Choosing item count >=5 gives approximately the same number of items as reported in paper
item_dict = {}
# Convert training sessions to sequences and renumber items to start from 1
def obtian_tra():
    train_ids = []
    train_seqs = []
    train_times = []
    train_dates = []
    item_ctr = 1
    for s, date in tra_sess:
        seq = sess_clicks[s]
        outseq = []
        for i in seq:
            if i in item_dict:
                outseq += [item_dict[i]]
            else:
                outseq += [item_ctr]
                item_dict[i] = item_ctr
                item_ctr += 1
        if len(outseq) < 2:  # Doesn't occur
            continue
        train_ids += [s]
        train_dates += [date]
        train_seqs += [outseq]
        train_times += [sess_times[s]]
    print('item_ctr')
    print(item_ctr)     # 43098, 37484
    return train_ids, train_dates, train_seqs, train_times


# Convert test sessions to sequences, ignoring items that do not appear in training set
def obtian_tes():
    test_ids = []
    test_seqs = []
    test_dates = []
    test_times = []
    for s, date in tes_sess:
        seq = sess_clicks[s]
        outseq = []
        for i in seq:
            if i in item_dict:
                outseq += [item_dict[i]]
        if len(outseq) < 2:
            continue
        test_ids += [s]
        test_dates += [date]
        test_seqs += [outseq]
        test_times += [sess_times[s]]
    return test_ids, test_dates, test_seqs, test_times


tra_ids, tra_dates, tra_seqs, tra_times = obtian_tra()
tes_ids, tes_dates, tes_seqs, tes_times = obtian_tes()

# data augmentation
def process_seqs(iseqs, idates, itimes):
    out_seqs = []
    out_dates = []
    out_times = []
    labs = []
    ids = []
    for id, seq, date, tim in zip(range(len(iseqs)), iseqs, idates, itimes):
        for i in range(1, len(seq)):
            tar = seq[-i]
            labs += [tar]
            out_seqs += [seq[:-i]]
            out_dates += [date]
            out_times += [tim[:-i]]
            ids += [id]
    return out_seqs, out_dates, out_times, labs, ids



tr_seqs, tr_dates, tr_times, tr_labs, tr_ids = process_seqs(tra_seqs, tra_dates, tra_times)
te_seqs, te_dates, te_times, te_labs, te_ids = process_seqs(tes_seqs, tes_dates, tes_times)

tra = (tr_seqs, tr_labs, tr_times)
tes = (te_seqs, te_labs, te_times)

print('train_test')
print(len(tr_seqs))
print(len(te_seqs))
print(tr_seqs[:3], tr_dates[:3], tr_labs[:3])
print(te_seqs[:3], te_dates[:3], te_labs[:3])
all = 0

for seq in tra_seqs:
    all += len(seq)
for seq in tes_seqs:
    all += len(seq)
print('max_sess_len', max(max([len(s) for s in tr_seqs]), max([len(s) for s in te_seqs])))
print('avg length: ', all/(len(tra_seqs) + len(tes_seqs) * 1.0))
print('all sessions:', all)

folder = "nowplaying"

# --- FINAL STEP: Save to the new, reduced directory ---
if not os.path.exists(folder):
    os.makedirs(folder)

def save_as_human_readable(data_list, filename):
    """
    Guarda una lista de secuencias (o listas) en un archivo de texto, 
    donde cada secuencia se convierte en una línea separada y los elementos 
    dentro de la secuencia se unen por un espacio.
    """
    with open(filename, 'w') as f:
        for sequence in data_list:
            # --- CORRECCIÓN ---
            # Verifica si el elemento no es iterable (ej. es un int o string simple)
            # y si no es un string (para evitar tratar strings como secuencias de caracteres).
            if not isinstance(sequence, (list, tuple)) or isinstance(sequence, str):
                # Si no es una lista/tupla (y no es un string), lo envuelve en una lista.
                # Ejemplo: 4 se convierte a [4]
                sequence = [sequence]
            # ------------------
            
            # Asegura que todos los elementos sean strings antes de unirlos
            line = ' '.join(map(str, sequence))
            f.write(line + '\n')
    print(f"Guardado exitosamente como texto legible en: {filename}")


save_as_human_readable(tra, f'{folder}/train_txt.txt')
save_as_human_readable(tes, f'{folder}/test_txt.txt')
save_as_human_readable(seq, f'{folder}/all_train_seq_txt.txt')

    
pickle.dump(tra, open(f'{folder}/train.txt', 'wb'))
pickle.dump(tes, open(f'{folder}/test.txt', 'wb'))
pickle.dump(tra_seqs, open(f'{folder}/all_train_seq.txt', 'wb'))

print(f'\nDone. New reduced dataset saved in the "{folder}" folder.')

# if not os.path.exists('Nowplaying'):
#     os.makedirs('Nowplaying')
# pickle.dump(tra, open('Nowplaying/train.txt', 'wb'))
# pickle.dump(tes, open('Nowplaying/test.txt', 'wb'))
# pickle.dump(tra_seqs, open('Nowplaying/all_train_seq.txt', 'wb'))

Generate short and long sessions

In [3]:
import argparse
import time
import csv
import pickle
import operator
import datetime
import os
dataset = '../datasets/nowplaying_procesed/raw/nowplaying.csv'

print("-- Starting @ %ss" % datetime.datetime.now())
with open(dataset, "r") as f:
    reader = csv.DictReader(f, delimiter='\t')
    sess_clicks = {}
    sess_times = {}
    sess_date = {}
    ctr = 0
    curid = -1
    curdate = None
    for data in reader:
        sessid = int(data['SessionId'])
        if curdate and not curid == sessid:
            date = curdate
            sess_date[curid] = date
        curid = sessid

        item = int(data['ItemId'])
        curdate = float(data['Time'])

        if sessid in sess_clicks:
            sess_clicks[sessid] += [item]
            sess_times[sessid] += [curdate]
        else:
            sess_clicks[sessid] = [item]
            sess_times[sessid] = [curdate]
        ctr += 1
    date = float(data['Time'])
    sess_date[curid] = date
print('ctr:', ctr)
print("-- Reading data @ %ss" % datetime.datetime.now())

# Filter out length 1 sessions
for s in list(sess_clicks):
    if len(sess_clicks[s]) == 1:
        del sess_clicks[s]
        del sess_date[s]
        del sess_times[s]

# Count number of times each item appears
iid_counts = {}
for s in sess_clicks:
    seq = sess_clicks[s]
    for iid in seq:
        if iid in iid_counts:
            iid_counts[iid] += 1
        else:
            iid_counts[iid] = 1

sorted_counts = sorted(iid_counts.items(), key=operator.itemgetter(1))

for s in list(sess_clicks):
    curseq = sess_clicks[s]
    curtime = sess_times[s]
    
    filseq = []
    filtime = []

    # 1. Iterar sobre los índices de la secuencia
    for i in range(len(curseq)):
      item_id = curseq[i]
      
      # 2. Aplicar la condición (El item debe aparecer 5 o más veces)
      if iid_counts[item_id] >= 5:
          # Si el item pasa el filtro, lo guardamos
          filseq.append(item_id)
          # Y OBLIGATORIAMENTE guardamos su tiempo asociado
          filtime.append(curtime[i])

    if len(filseq) < 2 or len(filseq) > 30:
        del sess_clicks[s]
        del sess_date[s]
        del sess_times[s]
    else:
        sess_clicks[s] = filseq
        sess_times[s] = filtime

# length = len(sess_clicks)
# for s in list(sess_clicks):
#     curseq = sess_clicks[s]
#     filseq = list(filter(lambda i: iid_counts[i] >= 5, curseq))
#     if len(filseq) < 2 or len(filseq) > 30:
#         del sess_clicks[s]
#         del sess_date[s]
#         del sess_times[s]
#     else:
#         sess_clicks[s] = filseq
#         sess_times[s] = filseq

# Split out test set based on dates
dates = list(sess_date.items())
maxdate = dates[0][1]

for _, date in dates:
    if maxdate < date:
        maxdate = date

# Two months for test
splitdate = maxdate - 60 * 86400

print('Splitting date', splitdate)      # Yoochoose: ('Split date', 1411930799.0)
tra_sess = filter(lambda x: x[1] < splitdate, dates)
tes_sess = filter(lambda x: x[1] > splitdate, dates)

# Sort sessions by date
tra_sess = sorted(tra_sess, key=operator.itemgetter(1))     # [(session_id, timestamp), (), ]
tes_sess = sorted(tes_sess, key=operator.itemgetter(1))     # [(session_id, timestamp), (), ]
print(len(tra_sess))    # 186670    # 7966257
print(len(tes_sess))    # 15979     # 15324
print(tra_sess[:3])
print(tes_sess[:3])
print("-- Splitting train set and test set @ %ss" % datetime.datetime.now())

# Choosing item count >=5 gives approximately the same number of items as reported in paper
item_dict = {}
# Convert training sessions to sequences and renumber items to start from 1
def obtian_tra():
    train_ids = []
    train_seqs = []
    train_dates = []
    train_times = []
    item_ctr = 1
    for s, date in tra_sess:
        seq = sess_clicks[s]
        outseq = []
        out_times = []
        for i in seq:
            if i in item_dict:
                outseq += [item_dict[i]]
            else:
                outseq += [item_ctr]
                item_dict[i] = item_ctr
                item_ctr += 1
        if len(outseq) < 2:  # Doesn't occur
            continue
        train_ids += [s]
        train_dates += [date]
        train_seqs += [outseq]
        train_times += [sess_times[s]]
    print(item_ctr)     # 43098, 37484
    return train_ids, train_dates, train_seqs, train_times


# Convert test sessions to sequences, ignoring items that do not appear in training set
def obtian_tes():
    test_ids = []
    test_seqs = []
    test_dates = []
    test_times = []
    for s, date in tes_sess:
        seq = sess_clicks[s]
        outseq = []
        for i in seq:
            if i in item_dict:
                outseq += [item_dict[i]]
        if len(outseq) < 2:
            continue
        test_ids += [s]
        test_dates += [date]
        test_seqs += [outseq]
        test_times += [sess_times[s]]
    return test_ids, test_dates, test_seqs, test_times


tra_ids, tra_dates, tra_seqs, tra_times = obtian_tra()
tes_ids, tes_dates, tes_seqs, tes_times = obtian_tes()

# data augmentation
def process_seqs(iseqs, idates, itimes, avg_len=7):
    short_item_set = set()
    long_item_set = set()
    s_out_seqs, s_out_dates, s_out_times, s_labs, s_ids = [], [], [], [], []
    l_out_seqs, l_out_dates, l_out_times, l_labs, l_ids = [], [], [], [], []
    
    for id, seq, date, tim in zip(range(len(iseqs)), iseqs, idates, itimes):
        if len(seq) - 1 <= avg_len:
            short_item_set.update(seq)
        else:
            short_item_set.update(seq[:avg_len+1])
            long_item_set.update(seq)

        for i in range(1, len(seq)):
            sub_seq = seq[:-i]
            if len(sub_seq) <= avg_len:
                tar = seq[-i]
                s_labs += [tar]
                s_out_seqs += [sub_seq]
                s_out_dates += [date]
                s_out_times += [tim[:-i]]
                s_ids += [id]

                # short_item_set.add(tar)
                # short_item_set.update(sub_seq)
            else:
                tar = seq[-i]
                l_labs += [tar]
                l_out_seqs += [sub_seq]
                l_out_dates += [date]
                l_out_times += [tim[:-i]]
                l_ids += [id]

                # long_item_set.add(tar)
                # long_item_set.update(sub_seq)
    return (s_out_seqs, s_out_dates, s_out_times, s_labs, s_ids, short_item_set), (l_out_seqs, l_out_dates, l_out_times, l_labs, l_ids, long_item_set)

def generate_final_sessions(tra_info, tes_info, folder):
    tr_seqs, tr_dates, tr_times, tr_labs, tr_ids, tr_items_unique = tra_info
    te_seqs, te_dates, te_times, te_labs, te_ids, te_items_unique = tes_info

    all_unique_items = tr_items_unique.union(te_items_unique)

    print(f"Total unique nodes: {len(all_unique_items)}")

    print("training sequences: ", len(tr_seqs))
    print("test sequences: ", len(te_seqs))
    
    tra = (tr_seqs, tr_labs, tr_times)
    tes = (te_seqs, te_labs, te_times)
    
    print('train_test')
    print(len(tr_seqs))
    print(len(te_seqs))
    print(tr_seqs[:3], tr_dates[:3], tr_labs[:3])
    print(te_seqs[:3], te_dates[:3], te_labs[:3])
    all = 0
    
    for seq in tra_seqs:
        all += len(seq)
    for seq in tes_seqs:
        all += len(seq)
    print('max_sess_len', max(max([len(s) for s in tr_seqs]), max([len(s) for s in te_seqs])))
    print('avg length: ', all/(len(tra_seqs) + len(tes_seqs) * 1.0))
    print('all clicks:', all)
    
    # --- FINAL STEP: Save to the new, reduced directory ---
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    def save_as_human_readable(data_list, filename):
        """
        Guarda una lista de secuencias (o listas) en un archivo de texto, 
        donde cada secuencia se convierte en una línea separada y los elementos 
        dentro de la secuencia se unen por un espacio.
        """
        with open(filename, 'w') as f:
            for sequence in data_list:
                # --- CORRECCIÓN ---
                # Verifica si el elemento no es iterable (ej. es un int o string simple)
                # y si no es un string (para evitar tratar strings como secuencias de caracteres).
                if not isinstance(sequence, (list, tuple)) or isinstance(sequence, str):
                    # Si no es una lista/tupla (y no es un string), lo envuelve en una lista.
                    # Ejemplo: 4 se convierte a [4]
                    sequence = [sequence]
                # ------------------
                
                # Asegura que todos los elementos sean strings antes de unirlos
                line = ' '.join(map(str, sequence))
                f.write(line + '\n')
        print(f"Guardado exitosamente como texto legible en: {filename}")
    
    
    save_as_human_readable(tra, f'{folder}/train_txt.txt')
    save_as_human_readable(tes, f'{folder}/test_txt.txt')
    save_as_human_readable(seq, f'{folder}/all_train_seq_txt.txt')
    
        
    pickle.dump(tra, open(f'{folder}/train.txt', 'wb'))
    pickle.dump(tes, open(f'{folder}/test.txt', 'wb'))
    pickle.dump(tra_seqs, open(f'{folder}/all_train_seq.txt', 'wb'))
    
    print(f'\nDone. New reduced dataset saved in the "{folder}" folder.')
    
    # if not os.path.exists('Nowplaying'):
    #     os.makedirs('Nowplaying')
    # pickle.dump(tra, open('Nowplaying/train.txt', 'wb'))
    # pickle.dump(tes, open('Nowplaying/test.txt', 'wb'))
    # pickle.dump(tra_seqs, open('Nowplaying/all_train_seq.txt', 'wb'))

short_info_tra, long_info_tra = process_seqs(tra_seqs, tra_dates, tra_times)
short_info_tes, long_info_tes = process_seqs(tes_seqs, tes_dates, tes_times)
generate_final_sessions(short_info_tra, short_info_tes, "nowplaying_short")
generate_final_sessions(long_info_tra, long_info_tes, "nowplaying_long")
print(len(item_dict.keys()))

-- Starting @ 2026-05-14 13:39:27.468061s
ctr: 1587776
-- Reading data @ 2026-05-14 13:39:36.713118s
Splitting date 1429749788.0
128077
14625
[(246034, 1388710627.0), (387474, 1388711024.0), (63365, 1388711048.0)]
[(296587, 1429750323.0), (106588, 1429750737.0), (1269, 1429751686.0)]
-- Splitting train set and test set @ 2026-05-14 13:39:43.881738s
60417
Total unique nodes: 60088
training sequences:  628325
test sequences:  68213
train_test
628325
68213
[[1], [3], [5, 6]] [1388710627.0, 1388711024.0, 1388711048.0] [2, 4, 7]
[[58991, 42356, 33620, 33619, 49142], [58991, 42356, 33620, 33619], [58991, 42356, 33620]] [1429750323.0, 1429750323.0, 1429750323.0] [5692, 49142, 33619]
max_sess_len 7
avg length:  7.419428154549791
all clicks: 1057684
Guardado exitosamente como texto legible en: nowplaying_short/train_txt.txt
Guardado exitosamente como texto legible en: nowplaying_short/test_txt.txt
Guardado exitosamente como texto legible en: nowplaying_short/all_train_seq_txt.txt

Done. New red